In [ ]:
!pip install datasets transformers manga109api Pillow torch torchvision ultralytics

from google.colab import userdata
from datasets import load_dataset
import zipfile
import manga109api
import os

from ultralytics import YOLO
from PIL import Image

In [ ]:
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="hal-utokyo/Manga109",
    repo_type="dataset",
    local_dir="/content/manga109"
)
print(path)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/content/manga109


In [ ]:
for root, dirs, files in os.walk("/content/manga109"):
    level = root.replace("/content/manga109", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only show files for top 2 levels
        for f in files[:5]:  # limit to 5 files per folder
            print(f"{indent}  {f}")

manga109/
  .gitattributes
  README.md
  Manga109_released_2023_12_07.zip
  .cache/
    huggingface/
      download/


In [ ]:
zip_path = "/content/manga109/Manga109_released_2023_12_07.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist()[:20])  # preview contents first

['Manga109_released_2023_12_07/', '__MACOSX/._Manga109_released_2023_12_07', 'Manga109_released_2023_12_07/books.txt', '__MACOSX/Manga109_released_2023_12_07/._books.txt', 'Manga109_released_2023_12_07/annotations.v2020.12.18/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2020.12.18', 'Manga109_released_2023_12_07/images/', '__MACOSX/Manga109_released_2023_12_07/._images', 'Manga109_released_2023_12_07/annotations.v2018.05.31/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2018.05.31', 'Manga109_released_2023_12_07/annotations/', '__MACOSX/Manga109_released_2023_12_07/._annotations', 'Manga109_released_2023_12_07/readme.txt', '__MACOSX/Manga109_released_2023_12_07/._readme.txt', 'Manga109_released_2023_12_07/annotations_COO/', '__MACOSX/Manga109_released_2023_12_07/._annotations_COO', 'Manga109_released_2023_12_07/annotations_Manga109Dialog/', '__MACOSX/Manga109_released_2023_12_07/._annotations_Manga109Dialog', 'Manga109_released_2023_12_07/annotations.v2020.12.18/M

In [ ]:
extract_path = "/content/manga109_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    members = [m for m in z.namelist() if not m.startswith("__MACOSX")]
    z.extractall(extract_path, members=members)

print("Done!")

Done!


In [ ]:
data_root = "/content/manga109_data/Manga109_released_2023_12_07"
api = manga109api.Parser(root_dir=data_root)

# List all books
print(api.books)

['ARMS', 'AisazuNihaIrarenai', 'AkkeraKanjinchou', 'Akuhamu', 'AosugiruHaru', 'AppareKappore', 'Arisa', 'BEMADER_P', 'BakuretsuKungFuGirl', 'Belmondo', 'BokuHaSitatakaKun', 'BurariTessenTorimonocho', 'ByebyeC-BOY', 'Count3DeKimeteAgeru', 'DollGun', 'Donburakokko', 'DualJustice', 'EienNoWith', 'EvaLady', 'EverydayOsakanaChan', 'GOOD_KISS_Ver2', 'GakuenNoise', 'GarakutayaManta', 'GinNoChimera', 'Hamlet', 'HanzaiKousyouninMinegishiEitarou', 'HaruichibanNoFukukoro', 'HarukaRefrain', 'HealingPlanet', 'HeiseiJimen', 'HighschoolKimengumi_vol01', 'HighschoolKimengumi_vol20', 'HinagikuKenzan', 'HisokaReturns', 'JangiriPonpon', 'JijiBabaFight', 'Joouari', 'Jyovolley', 'KarappoHighschool', 'KimiHaBokuNoTaiyouDa', 'KoukouNoHitotachi', 'KuroidoGanka', 'KyokugenCyclone', 'LancelotFullThrottle', 'LoveHina_vol01', 'LoveHina_vol14', 'MAD_STONE', 'MadouTaiga', 'MagicStarGakuin', 'MagicianLoad', 'MariaSamaNihaNaisyo', 'MayaNoAkaiKutsu', 'MemorySeijin', 'MeteoSanStrikeDesu', 'MiraiSan', 'MisutenaideDaisy'

In [ ]:
import os
from PIL import Image

LABEL2ID = {"body": 0, "text": 1}

def convert_all_annotations(api, data_root, label_out_root):
    image_dir = os.path.join(data_root, "images")

    for book in api.books:
        ann = api.get_annotation(book=book)

        for page in ann['page']:
            page_index = page['@index']
            W = page['@width']
            H = page['@height']

            img_filename = f"{page_index:03d}.jpg"
            img_path = os.path.join(image_dir, book, img_filename)
            if not os.path.exists(img_path):
                continue

            lines = []
            for tag, label_id in LABEL2ID.items():
                for item in page[tag]:
                    x1 = item['@xmin']
                    y1 = item['@ymin']
                    x2 = item['@xmax']
                    y2 = item['@ymax']

                    cx = ((x1 + x2) / 2) / W
                    cy = ((y1 + y2) / 2) / H
                    w  = (x2 - x1) / W
                    h  = (y2 - y1) / H
                    lines.append(f"{label_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            label_out = os.path.join(label_out_root, book)
            os.makedirs(label_out, exist_ok=True)
            with open(os.path.join(label_out, img_filename.replace(".jpg", ".txt")), "w") as f:
                f.write("\n".join(lines))

    print("Annotation conversion done!")

label_out_root = "/content/manga109_labels"
convert_all_annotations(api, data_root, label_out_root)

Annotation conversion done!


In [ ]:
# Verify a label file looks right
import random
book = random.choice(api.books)
label_file = f"/content/manga109_labels/{book}/002.txt"
if os.path.exists(label_file):
    with open(label_file) as f:
        print(f.read())

0 0.231258 0.824359 0.173519 0.349573
0 0.385429 0.812821 0.115478 0.191453
0 0.128476 0.517094 0.169891 0.211966
0 0.092503 0.859402 0.085852 0.164957
1 0.224002 0.465812 0.035671 0.109402
1 0.440750 0.708974 0.008464 0.074359
1 0.130895 0.716239 0.053809 0.109402
1 0.290810 0.726496 0.016929 0.102564


In [ ]:
import shutil, yaml, os

data_root = "/content/manga109_data/Manga109_released_2023_12_07"
image_dir = os.path.join(data_root, "images")
label_out_root = "/content/manga109_labels"
dataset_root = "/content/manga109_dataset"

# Train/val split (80/20)
all_books = api.books
split = int(len(all_books) * 0.8)
train_books = all_books[:split]
val_books = all_books[split:]

print(f"Train: {len(train_books)} books, Val: {len(val_books)} books")

# Copy images and labels into dataset folder
for split_name, books in [("train", train_books), ("val", val_books)]:
    img_split_dir = os.path.join(dataset_root, "images", split_name)
    lbl_split_dir = os.path.join(dataset_root, "labels", split_name)
    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(lbl_split_dir, exist_ok=True)

    for book in books:
        src_img = os.path.join(image_dir, book)
        src_lbl = os.path.join(label_out_root, book)
        dst_img = os.path.join(img_split_dir, book)
        dst_lbl = os.path.join(lbl_split_dir, book)

        if os.path.exists(src_img) and not os.path.exists(dst_img):
            shutil.copytree(src_img, dst_img)
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            shutil.copytree(src_lbl, dst_lbl)

print("Dataset split done!")

Train: 87 books, Val: 22 books
Dataset split done!


In [ ]:

import os, shutil, yaml, json, time
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import manga109api

data_root    = "/content/manga109_data/Manga109_released_2023_12_07"
dataset_root = "/content/manga109_dataset"
api          = manga109api.Parser(root_dir=data_root)

dataset_cfg = {
    "path":  dataset_root,
    "train": "images/train",
    "val":   "images/val",
    "nc":    2,
    "names": {0: "body", 1: "text"},
}
yaml_path = os.path.join(dataset_root, "manga109.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f)
print(f"Dataset YAML written → {yaml_path} ✓")


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        B, C, _, _ = x.shape
        avg = self.mlp(self.avg_pool(x).view(B, C))
        mx  = self.mlp(self.max_pool(x).view(B, C))
        w   = self.sigmoid(avg + mx).view(B, C, 1, 1)
        return x * w


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        w   = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * w


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x


class CBAMWrapper(nn.Module):

    def __init__(self, c2f_layer, channels):
        super().__init__()
        self.c2f  = c2f_layer
        self.cbam = CBAM(channels)
        self.f    = c2f_layer.f
        self.i    = c2f_layer.i
        self.type = "CBAMWrapper"

    def forward(self, x):
        return self.cbam(self.c2f(x))


#ai was used in these modules (this feature was not integrating correctly, and we used claude + chatgpt in order to try to fix it)
# ══════════════════════════════════════════════════════════════════
#  PATCHED TRAINER
# ══════════════════════════════════════════════════════════════════

class PatchedTrainer(DetectionTrainer):
    def __init__(self, patched_model, overrides=None):
        self._patched_model = patched_model
        super().__init__(overrides=overrides)

    def get_model(self, cfg=None, weights=None, verbose=True):
        self._patched_model.nc = self.data['nc']
        return self._patched_model


# ══════════════════════════════════════════════════════════════════
#  BUILD AND PATCH MODEL
# ══════════════════════════════════════════════════════════════════

base     = YOLO("yolov8s.pt")
detector = base.model

# Neck C2f layers and their output channels
# From earlier inspection:
#   Layer 12: C2f output = 256  (P4 after first upsample+concat)
#   Layer 15: C2f output = 128  (P3 — smallest scale, largest spatial)
#   Layer 18: C2f output = 256  (P4 — medium scale)
#   Layer 21: C2f output = 512  (P5 — largest scale, smallest spatial)

NECK_C2F = {
    12: 256,
    15: 128,
    18: 256,
    21: 512,
}

for layer_idx, out_ch in NECK_C2F.items():
    old = detector.model[layer_idx]
    new = CBAMWrapper(old, channels=out_ch)
    detector.model[layer_idx] = new
    print(f"Layer {layer_idx} → CBAMWrapper(channels={out_ch}) ✓")

# ── Verify forward pass ────────────────────────────────────────────
detector.cuda()
dummy = torch.randn(1, 3, 640, 640).cuda()
with torch.no_grad():
    detector(dummy)
print("Forward pass OK ✓")

# ── Layer table ───────────────────────────────────────────────────
print("\nLayer verification:")
for i, layer in enumerate(detector.model):
    marker = " ← CUSTOM" if type(layer).__name__ == 'CBAMWrapper' else ""
    print(f"  Layer {i:2d} | {type(layer).__name__:22s}{marker}")

total = sum(p.numel() for p in detector.parameters())
print(f"\nTotal params: {total/1e6:.2f}M")

# ══════════════════════════════════════════════════════════════════
#  TRAIN
# ══════════════════════════════════════════════════════════════════

args = dict(
    data         = yaml_path,
    epochs       = 40,              # ← was 20
    imgsz        = 640,
    batch        = 8,
    name         = "manga109_v4_cbam_neck_40ep",
    project      = "/content/runs",
    device       = 0,
    optimizer    = "AdamW",
    lr0          = 1e-3,            # ← bump up slightly for longer training
    warmup_epochs= 3,
    cos_lr       = True,            # ← add cosine LR decay
    hsv_s        = 0.5,
    mixup        = 0.1,
    model        = "yolov8s.pt",
    cache        = False,           # ← changed from True to save RAM
)

trainer = PatchedTrainer(patched_model=detector, overrides=args)
trainer.train()
print("Training complete ✓")

Dataset YAML written → /content/manga109_dataset/manga109.yaml ✓
Layer 12 → CBAMWrapper(channels=256) ✓
Layer 15 → CBAMWrapper(channels=128) ✓
Layer 18 → CBAMWrapper(channels=256) ✓
Layer 21 → CBAMWrapper(channels=512) ✓
Forward pass OK ✓

Layer verification:
  Layer  0 | Conv                  
  Layer  1 | Conv                  
  Layer  2 | C2f                   
  Layer  3 | Conv                  
  Layer  4 | C2f                   
  Layer  5 | Conv                  
  Layer  6 | C2f                   
  Layer  7 | Conv                  
  Layer  8 | C2f                   
  Layer  9 | SPPF                  
  Layer 10 | Upsample              
  Layer 11 | Concat                
  Layer 12 | CBAMWrapper            ← CUSTOM
  Layer 13 | Upsample              
  Layer 14 | Concat                
  Layer 15 | CBAMWrapper            ← CUSTOM
  Layer 16 | Conv                  
  Layer 17 | Concat                
  Layer 18 | CBAMWrapper            ← CUSTOM
  Layer 19 | Conv            

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/40      3.61G      1.258     0.9875      1.124        224        640: 100% ━━━━━━━━━━━━ 1066/1066 3.6it/s 4:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.0it/s 32.5s
                   all       2077      59398      0.838      0.767      0.852       0.58

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/40      3.61G      1.072     0.8081      1.087        336        640: 0% ──────────── 0/1066  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/40      3.61G      1.167     0.8578      1.087        397        640: 100% ━━━━━━━━━━━━ 1066/1066 3.8it/s 4:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.1it/s 31.5s
                   all       2077      59398      0.867      0.809      0.882       0.61

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/40      3.61G      1.204     0.8368       1.11        385        640: 0% ──────────── 0/1066  0.2s

...

      38/40      5.21G     0.8282      0.504     0.9388        144        640: 100% ━━━━━━━━━━━━ 1066/1066 4.3it/s 4:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.1it/s 31.8s
                   all       2077      59398      0.908      0.862      0.928      0.684

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      39/40      5.21G     0.8834     0.5629     0.9392        295        640: 0% ──────────── 0/1066  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      39/40      5.21G     0.8239     0.5022     0.9375        188        640: 100% ━━━━━━━━━━━━ 1066/1066 4.3it/s 4:10
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.3it/s 30.4s
                   all       2077      59398      0.908      0.864      0.928      0.685

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      40/40      5.21G      0.821     0.4213     0.9175        218        640: 0% ──────────── 0/1066  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/40      5.21G     0.8227     0.5007     0.9363        133        640: 100% ━━━━━━━━━━━━ 1066/1066 4.3it/s 4:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 4.4it/s 29.8s
                   all       2077      59398      0.909      0.864      0.928      0.684

40 epochs completed in 3.354 hours.
Optimizer stripped from /content/runs/manga109_v4_cbam_neck_40ep/weights/last.pt, 22.7MB
Optimizer stripped from /content/runs/manga109_v4_cbam_neck_40ep/weights/best.pt, 22.7MB

Validating /content/runs/manga109_v4_cbam_neck_40ep/weights/best.pt...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 104 layers, 11,209,360 parameters, 0 gradients, 28.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 130/130 3.7it/s 35.3s
                   all       2077      59398      0.908      0.864      0.92

In [ ]:

# ══════════════════════════════════════════════════════════════════
#  EVALUATE
# ══════════════════════════════════════════════════════════════════

weights_path  = os.path.join(trainer.save_dir, "weights", "best.pt")
ckpt          = torch.load(weights_path, map_location='cpu', weights_only=False)
detector_eval = (ckpt.get('ema') or ckpt['model']).float().cuda()

eval_yolo       = YOLO("yolov8s.pt")
eval_yolo.model = detector_eval
metrics         = eval_yolo.val(data=yaml_path, imgsz=640, batch=8)

map50             = metrics.box.map50
map5095           = metrics.box.map
map50_per_class   = metrics.box.ap50
map5095_per_class = metrics.box.ap
precision         = metrics.box.mp
recall            = metrics.box.mr
f1                = 2 * (precision * recall) / (precision + recall + 1e-8)

print(f"\nmAP50:     {map50:.4f}")
print(f"mAP50-95:  {map5095:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

# ── Inference speed ───────────────────────────────────────────────
sample_img = os.path.join(data_root, "images", api.books[0], "002.jpg")
for _ in range(3):
    eval_yolo(sample_img, verbose=False)

times = []
for _ in range(50):
    start = time.perf_counter()
    eval_yolo(sample_img, verbose=False)
    times.append((time.perf_counter() - start) * 1000)

avg_ms = np.mean(times)
fps    = 1000 / avg_ms
print(f"Latency: {avg_ms:.2f} ms | FPS: {fps:.1f}")

# ── Save results ──────────────────────────────────────────────────
model_size_mb = os.path.getsize(weights_path) / (1024 ** 2)
param_count   = sum(p.numel() for p in detector_eval.parameters())

results = {
    "model":         "YOLOv8s_V4_CBAM_Neck",
    "mAP50":         round(map50, 4),
    "mAP50-95":      round(map5095, 4),
    "precision":     round(precision, 4),
    "recall":        round(recall, 4),
    "f1":            round(f1, 4),
    "AP50_body":     round(map50_per_class[0], 4),
    "AP50_text":     round(map50_per_class[1], 4),
    "AP5095_body":   round(map5095_per_class[0], 4),
    "AP5095_text":   round(map5095_per_class[1], 4),
    "latency_ms":    round(avg_ms, 2),
    "fps":           round(fps, 1),
    "model_size_mb": round(model_size_mb, 2),
    "params_M":      round(param_count / 1e6, 2),
}

print("\n── Summary ──")
for k, v in results.items():
    print(f"  {k}: {v}")

with open("/content/results_v4_cbam_neck_40ep.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → /content/results_v4_cbam_neck_40ep.json")

# ── Save to Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/drive')

shutil.copytree(
    str(trainer.save_dir),
    "/drive/MyDrive/manga109_v4_cbam_neck_40ep",
    dirs_exist_ok=True
)
shutil.copy(
    "/content/results_v4_cbam_neck_40ep.json",
    "/drive/MyDrive/CV-Comic-Project/results_v4_cbam_neck_40ep.json"
)
print("Saved to Drive ✓")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 104 layers, 11,209,360 parameters, 0 gradients, 28.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3017.0±642.5 MB/s, size: 373.1 KB)
val: Scanning /content/manga109_dataset/labels/val/TapkunNoTanteisitsu.cache... 2077 images, 94 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2077/2077 726.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 260/260 6.7it/s 38.6s
                   all       2077      59398      0.908      0.864      0.928      0.686
                  body       1979      31178      0.894      0.816      0.907      0.658
                  text       1962      28220      0.923      0.913      0.949      0.715
Speed: 1.1ms preprocess, 7.9ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val

mAP50:     0.9284
mAP50-95:  0.6864
Precision: 0.9084
Recall: